# Assignment 7

## Submit as an HTML file

### Print your name below

In [22]:
print("Joanna Yao")

Joanna Yao


### Import the "pandas" "numpy" and "statsmodels.formula.api" libraries

In [2]:
# Write your answer here:

import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

#### In the code chunk below read the CSV file named `results.csv` in the `data` <br> folder and print the first 5 rows of the dataset. Browse the dataset.

In [3]:
data=pd.read_csv("data/results.csv")
data.head()

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1


### (a)  Check Column Types and Data Cleaning

- Use the function .dtypes to get the column types
- Identify which columns have data types that might need conversion
- The 'milliseconds' column contains string values that should be numeric. Create a new column called 'race_time_ms' that:
    - Converts the column to a numeric data type
    - Replaces any non-numeric values with NaN

In [25]:
# Write your answer here

# got column types
data.dtypes

# The columns that would need conversion are "number", "position", "positionText", "time", "milliseconds", "fastestLap", "rank", "fastestLapTime", and "fastestLapSpeed". 
data['race_time_ms'] = pd.to_numeric(data['milliseconds'],errors='coerce')


### (b) Create Categorical Variables

- Create a new column called 'finish_category' that categorizes the race finish positions as follows:
    - Positions 1-3: 'Podium'
    - Positions 4-10: 'Points'
    - Positions 11-20: 'Midfield'
    - Positions >20: 'Backmarker'

Hint: Use the pd.cut() function

In [27]:
# Write your answer here

# make sure bin edges match -> 5 labels for 6 bin edges. 
data['position']=pd.to_numeric(data['milliseconds'],errors='coerce')
bins_j=[0,3,10,20,float('inf')]
labels_j=["Podium","Points","Midfield","Backmarker"]

data['finish_category']=pd.cut(data['positionOrder'],bins=bins_j,right=True,labels=labels_j)
data

,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId,race_time_ms,finish_category
0,1,18,1,1,22,1,5690616.0,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1,5690616.0,Podium
1,2,18,2,2,3,5,5696094.0,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1,5696094.0,Podium
2,3,18,3,3,7,7,5698779.0,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1,5698779.0,Podium
3,4,18,4,4,5,11,5707797.0,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1,5707797.0,Points
4,5,18,5,1,23,3,5708630.0,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1,5708630.0,Points
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25835,25841,1096,854,210,47,12,NaN,16,16,0.0,57,\N,\N,39,12,1:29.833,211.632,11,NaN,Midfield
25836,25842,1096,825,210,20,16,NaN,17,17,0.0,57,\N,\N,40,20,1:31.158,208.556,11,NaN,Midfield
25837,25843,1096,1,131,44,5,NaN,18,18,0.0,55,\N,\N,42,11,1:29.788,211.738,9,NaN,Midfield
25838,25844,1096,849,3,6,20,NaN,19,19,0.0,55,\N,\N,45,14,1:30.309,210.517,130,NaN,Midfield


### (c) Calculate Race Duration
- For rows where 'milliseconds' is available, create a new column <br>
'race_duration_minutes' that converts milliseconds to minutes by dividing <br>
by (1000*60).
- Display the average race duration by 'constructorId' for the top 5 <br>
constructors with the shortest average race times

In [60]:
# Write your answer here

# make column
data['race_duration_minutes']=data['race_time_ms']/(1000*60)
data

# make display
durationagg=(data.groupby('constructorId').agg(mean_race_time=('race_duration_minutes','mean')))
durationagg
sort_mean_race_time=durationagg.sort_values(by="mean_race_time",ascending=True)
ds_1=sort_mean_race_time.iloc[0:5]
ds_1


,mean_race_time
constructorId,
35,76.710777
29,77.604125
41,87.046767
16,89.428828
53,89.658852


### (d) Driver Performance Analysis

- Calculate the following statistics for each driver, grouped by 'driverId':
    - Average finishing position
    - Total points
    - Number of races completed
    - Best finishing position

- Sort the results by total points in descending order
- Display the top 10 drivers based on total points

In [4]:
# Write your answer here

# calculate statistics
driveragg=(data.groupby('driverId').agg(
    average_finishing_position=('positionOrder','mean'),
    total_points=('points','sum'),
    number_of_races_completed=('driverId',len),
    best_finishing_position=('positionOrder','min')))
driveragg

# sort results + display
sort_points=driveragg.sort_values(by="total_points",ascending=False)
sort_points
ds_2=sort_points.iloc[0:10]
ds_2

,average_finishing_position,total_points,number_of_races_completed,best_finishing_position
driverId,,,,
1,4.787097,4396.5,310,1
20,7.093333,3098.0,300,1
4,8.494413,2061.0,358,1
830,6.533742,1983.5,163,1
8,8.491477,1873.0,352,1
822,7.601990,1778.0,201,1
3,8.252427,1594.5,206,1
30,6.879870,1566.0,308,1
817,9.883621,1307.0,232,1


### (e) Linear Regression
Create a linear regression model that predicts 'points' based on 'grid' (starting position) and 'laps' completed <br>
Use the following steps:

- Clean the data to remove any non-numeric values and missing values
- Create the regression formula using smf.ols 
- Display the summary of the regression model using model.summary()

What is the predicted points for a driver starting in position 3 and completing 55 laps?

Hint: Use ```.dropna()''' to remove missing values from the points, grid, and laps <br>
variables.

In [86]:
# Write your answer here

# creating the linear regression model + cleaning
ds_3=data[['points','grid','laps']]
ds_3=ds_3.dropna()

# regression formula
model=smf.ols(formula='points ~ grid + laps',data=ds_3).fit(cov_type="HC1")
print(summary_col(model,stars=True))

# regression formula: points = 2.5841 - (.2248)*(grid) + (.0393)(laps)
predicted=2.5841-(.2248)*(3)+(.0393)*(55)
print(str(predicted))

# The predicted points is 4.0712 points. 



                 points  
-------------------------
Intercept      2.5841*** 
               (0.0539)  
grid           -0.2248***
               (0.0036)  
laps           0.0393*** 
               (0.0008)  
R-squared      0.2146    
R-squared Adj. 0.2145    
Standard errors in
parentheses.
* p<.1, ** p<.05,
***p<.01
4.0712
